# LandmarkDiff Quickstart

Predict post-operative facial appearance from a single photograph.

This notebook walks through:
1. Loading the pipeline
2. Running a basic rhinoplasty prediction
3. Comparing original vs predicted
4. Trying different procedures and intensities
5. Using clinical flags

**Requirements:** `pip install -e .` from the repo root. GPU optional -- TPS mode runs on CPU.

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import urllib.request

## 1. Load a face image

Upload your own photo or grab a sample. Needs a clear frontal face with decent lighting.

In [ ]:
# download a sample face if you don't have one handy
sample_url = "https://raw.githubusercontent.com/NVlabs/ffhq-dataset/master/thumbnails128x128/00000.png"
sample_path = Path("sample_face.png")

if not sample_path.exists():
    urllib.request.urlretrieve(sample_url, sample_path)

img = Image.open(sample_path).convert("RGB").resize((512, 512))
img_array = np.array(img)

plt.figure(figsize=(4, 4))
plt.imshow(img_array)
plt.title("input")
plt.axis("off")
plt.show()

## 2. Load the pipeline

TPS mode uses thin-plate spline warping and runs on CPU with no model downloads needed.
For photorealistic output, use `mode="controlnet"` (requires GPU + ~6GB VRAM).

In [ ]:
from landmarkdiff.inference import LandmarkDiffPipeline

# CPU mode -- no model downloads, runs anywhere
pipe = LandmarkDiffPipeline(mode="tps")
pipe.load()

print(f"pipeline loaded, mode: {pipe.mode}")

## 3. Run a basic rhinoplasty prediction

The `generate` method takes an image array, a procedure name, and an intensity (0-100).
It returns a dict with the output image and all intermediate results.

In [ ]:
result = pipe.generate(
    img_array,
    procedure="rhinoplasty",
    intensity=50,
)

print("result keys:", list(result.keys()))

## 4. Compare original vs predicted

Side-by-side view of the input and the TPS-warped output.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(img_array)
axes[0].set_title("original")
axes[0].axis("off")

axes[1].imshow(result["output"])
axes[1].set_title("rhinoplasty @ 50%")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Try different procedures and intensities

LandmarkDiff ships with 6 procedure presets. Each one targets different anatomical landmarks.
The intensity parameter goes from 0 (no change) to 100 (maximum deformation).

In [ ]:
procedures = ["rhinoplasty", "blepharoplasty", "rhytidectomy", "orthognathic", "brow_lift", "mentoplasty"]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, proc in enumerate(procedures):
    r = pipe.generate(img_array, procedure=proc, intensity=60)
    axes[i].imshow(r["output"])
    axes[i].set_title(f"{proc} @ 60%")
    axes[i].axis("off")

plt.suptitle("all procedures at 60% intensity", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# intensity sweep for rhinoplasty: 20%, 40%, 60%, 80%, 100%
intensities = [20, 40, 60, 80, 100]

fig, axes = plt.subplots(1, len(intensities), figsize=(20, 4))

for i, level in enumerate(intensities):
    r = pipe.generate(img_array, procedure="rhinoplasty", intensity=level)
    axes[i].imshow(r["output"])
    axes[i].set_title(f"{level}%")
    axes[i].axis("off")

plt.suptitle("rhinoplasty intensity sweep", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Inspect intermediate outputs

The result dict contains the landmarks, conditioning wireframe, and surgical mask
from each stage of the pipeline.

In [ ]:
result = pipe.generate(img_array, procedure="rhinoplasty", intensity=50)

# landmarks are FaceLandmarks objects -- pixel_coords is a property, not a method
original_lm = result["landmarks_original"]
deformed_lm = result["landmarks_manipulated"]

print(f"original landmarks shape: {original_lm.pixel_coords.shape}")
print(f"deformed landmarks shape: {deformed_lm.pixel_coords.shape}")

# show conditioning and mask if available
panels = []
titles = []
if "conditioning" in result and result["conditioning"] is not None:
    panels.append(result["conditioning"])
    titles.append("conditioning wireframe")
if "mask" in result and result["mask"] is not None:
    panels.append(result["mask"])
    titles.append("surgical mask")

if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
    if len(panels) == 1:
        axes = [axes]
    for ax, panel, title in zip(axes, panels, titles):
        ax.imshow(panel, cmap="gray" if panel.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 7. Using clinical flags

LandmarkDiff handles clinical edge cases that affect how deformations are applied:
- **Vitiligo** -- preserves depigmented patches
- **Bell's palsy** -- disables deformation on the paralyzed side
- **Keloid-prone skin** -- softens mask boundaries near incision sites
- **Ehlers-Danlos** -- widens RBF radius for hypermobile tissue

In [ ]:
from landmarkdiff.clinical import ClinicalFlags

# example: patient with keloid-prone skin in the nasal region
flags = ClinicalFlags(
    keloid_prone=True,
    keloid_regions=["nose"],
)

result_clinical = pipe.generate(
    img_array,
    procedure="rhinoplasty",
    intensity=50,
    clinical_flags=flags,
)

# compare standard vs clinical-aware output
result_standard = pipe.generate(
    img_array,
    procedure="rhinoplasty",
    intensity=50,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(result_standard["output"])
axes[0].set_title("standard")
axes[0].axis("off")
axes[1].imshow(result_clinical["output"])
axes[1].set_title("with keloid flags")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Bell's palsy example -- left side affected
flags_palsy = ClinicalFlags(
    bells_palsy=True,
    bells_palsy_side="left",
)

result_palsy = pipe.generate(
    img_array,
    procedure="rhytidectomy",
    intensity=60,
    clinical_flags=flags_palsy,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_array)
axes[0].set_title("original")
axes[0].axis("off")
axes[1].imshow(result_palsy["output"])
axes[1].set_title("rhytidectomy with Bell's palsy (left)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 8. GPU pipeline (optional)

For photorealistic results, use the ControlNet mode. This requires a GPU with at least 6GB VRAM
and will download ~6GB of model weights on first run.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("no GPU detected -- skip this cell or switch to a GPU runtime")
    print("the TPS results above still work fine on CPU")
else:
    gpu_pipe = LandmarkDiffPipeline(mode="controlnet", device="cuda")
    gpu_pipe.load()

    result_gpu = gpu_pipe.generate(
        img_array,
        procedure="rhinoplasty",
        intensity=50,
        num_inference_steps=30,
        guidance_scale=7.5,
        seed=42,
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_array)
    axes[0].set_title("input")
    axes[0].axis("off")
    axes[1].imshow(result_gpu["conditioning"])
    axes[1].set_title("conditioning")
    axes[1].axis("off")
    axes[2].imshow(result_gpu["output"])
    axes[2].set_title("controlnet prediction")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

    if result_gpu.get("identity_check"):
        print(f"identity similarity: {result_gpu['identity_check']:.3f}")

## Next steps

- Try uploading your own photos and adjusting intensities
- Define a [custom procedure](https://github.com/dreamlessx/LandmarkDiff-public/blob/main/docs/tutorials/custom_procedures.md) with your own landmark targets
- Run the [Gradio demo](https://github.com/dreamlessx/LandmarkDiff-public#gradio-web-demo) for a full interactive UI
- Check the [API docs](https://github.com/dreamlessx/LandmarkDiff-public/tree/main/docs/api) for detailed module reference
- See the [CHANGELOG](https://github.com/dreamlessx/LandmarkDiff-public/blob/main/CHANGELOG.md) for recent updates